### Подготовка окружения

Здесь я повторяю тот же шаг, что и в ноутбуке генерации: поднимаю окружение под PySpark в Colab (Java + `pyspark`), затем создаю `SparkSession`.
Дальше ноутбук читает RAW JSON-партиции из Google Drive и собирает витрины (marts).


In [7]:
from google.colab import drive
drive.mount('/content/gdrive')

Mounted at /content/gdrive


In [2]:
# ✅ Рабочая установка PySpark в Google Colab (актуально для Python 3.12)
# Не скачиваем Spark вручную и не используем findspark — ставим pyspark через pip.
!apt-get -qq update
!apt-get -qq install -y openjdk-17-jdk-headless > /dev/null
!pip -q install pyspark==3.5.1

from pyspark.sql import SparkSession

spark = (SparkSession.builder
         .master("local[*]")
         .appName("pet_project_create_mart")
         .getOrCreate())

spark.conf.set("spark.sql.repl.eagerEval.enabled", True)

# Проверка
spark.version


W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 317.0/317.0 MB 4.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 200.5/200.5 kB 10.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dataproc-spark-connect 1.0.1 requires pyspark[connect]~=4.0.0, but you have pyspark 3.5.1 which is incompatible.


'3.5.1'

Объявление схемы данных для дальнейшего считывания JSON файлов и составления Data Frame.

In [16]:
from pyspark.sql import functions as F
from pyspark.sql import types as T
from pyspark.sql.window import Window



schema = T.StructType([
    T.StructField("inn", T.StringType(), True),
    T.StructField("raw_cookie", T.ArrayType(T.MapType(T.StringType(),
                                                      T.StringType()))),
    T.StructField("event_type", T.StringType(), True),
    T.StructField("event_action", T.StringType(), True),
    T.StructField("data_value", T.StringType(), True),
    T.StructField("geocountry", T.StringType(), True),
    T.StructField("city", T.StringType(), True),
    T.StructField("user_os", T.StringType(), True),
    T.StructField("systemlanguage", T.StringType(), True),
    T.StructField("geoaltitude", T.StringType(), True),
    T.StructField("meta_platform", T.StringType(), True),
    T.StructField("screensize", T.StringType(), True),
    T.StructField("timestampcolumn", T.DateType(), True)
                       ])

Объявление обязательных переменных, которые дальше использую в функциях

In [8]:
import os

PATH_TO_FILES = "/content/gdrive/MyDrive/data/json/"
DICT_MATCH_CODE = {"IOS": "IDFA", "Android": "GAID"}
filenames = sorted(os.listdir(PATH_TO_FILES), reverse=True)

Функция чтения RAW JSON-партиций

In [10]:
def read_file_json(file_name, schema_json_file):
 return spark.read.format("json") \
            .load(f"{file_name}", schema=schema_json_file)

Функция объединения данных JSON файлов и удаления дубликатов в них. Удаление производится с помощью оконной функции rank.

In [11]:
def union_and_dropduplicate(df1, df2):
    return (df1.union(df2).withColumn('rank', F.rank()
    .over(Window.partitionBy('inn')
    .orderBy(F.desc('timestampcolumn'))))
                          .filter('rank = 1')
                          .drop('rank'))


Функция считывания и создания таблицы из сгенерированных файлов

In [12]:
def initial_data(schema_json_file: T.StructType, count: int = len(filenames)):
    blank_df = spark.createDataFrame([], schema=schema_json_file)
    for i in filenames[:count]:
        json_file = read_file_json(f"{PATH_TO_FILES}{i}", schema_json_file)
        blank_df = union_and_dropduplicate(blank_df, json_file)
    return blank_df

Функция, предоставляющая возможность получить значение словоря по его ключу

In [13]:
def search_values_from_maptype(col_name, key):
    return F.expr(f"filter({col_name}, x -> x.key='{key}')")[0]["value"]

Функция преобразования соответствия ОС. Используется в витрине "G"

In [14]:
@F.udf
def match_code(x):
    return DICT_MATCH_CODE.get(x, None)

Чтобы не обращаться постоянно к функции, считывание данных занесем в переменную "initial_data_df"

In [17]:
initial_data_df = initial_data(schema).select("*")

# Создание витрины "A".
В общем и целом данная витрина - это наша сгенерированная таблица, только без поля "INN".

In [18]:
data_mart_a = initial_data_df.drop('inn')
data_mart_a.show(5, truncate=False)

+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+----------+------------+----------------------------------------------------------------+-------------------+--------+-------+--------------+-----------+-------------+----------+---------------+
|raw_cookie                                                                                                                                                                                                                                                                                                                                                                                    

# Создание витрины "В".
В данной витрине необходимо найти все "INN" пользователей. Для поля "ID" в данной витрине была примена оконная функция, с целью показать свои знания, в последующих витриниках оконных функций использовано не будет из-за ее долгой работы при сортировке данных.

In [19]:
data_mart_b = initial_data_df.select(
    F.row_number().over(
        Window.orderBy(
            F.col('inn').desc()
            )
        ).alias("id"),
        "inn"
  )
data_mart_b.show(5)


+---+------------+
| id|         inn|
+---+------------+
|  1|999705889885|
|  2|998933315633|
|  3|998492595671|
|  4|997525328167|
|  5|997099884215|
+---+------------+
only showing top 5 rows



# Создание витрины "С"
В данной витрине необходимо собрать куки сайта. Для это использовалась функция
"pyspark.sql.functions.expr", в которую передовалось сгенирированное значение из функции "search_values_from_cookie". Для генерации ID используется метод "monotonically_increasing_id". В последующий витринах используются эти функции.

In [20]:
data_mart_c = initial_data_df.select(
  (F.monotonically_increasing_id() + 1).alias("id"),
  search_values_from_maptype("raw_cookie", "_sa_cookie_a").alias("sa_cookie_a"))
data_mart_c.show(5)

+---+--------------------+
| id|         sa_cookie_a|
+---+--------------------+
|  1|SA1.dc9a95a7-b909...|
|  2|SA1.e9f10edb-0da9...|
|  3|SA1.6d3bddd8-94ad...|
|  4|SA1.582db06f-2639...|
|  5|SA1.781f4c22-7e50...|
+---+--------------------+
only showing top 5 rows



# Создание витрины "D"
Тоже самое что и в витрине "В". Подразумевается что запущен файл "Generation_files.ipynb", который к этому моменту, должен сгенерировать дополнительные JSON файлы, которые не пересекутся с витриной "В" при объединении.

In [21]:
data_mart_d = initial_data_df.select(
    (F.monotonically_increasing_id() + 1).alias("id"),
    "inn"
)
data_mart_d.show(5, truncate=False)

+---+------------+
|id |inn         |
+---+------------+
|1  |000173638241|
|2  |001137618533|
|3  |001671856037|
|4  |001718749268|
|5  |001737951886|
+---+------------+
only showing top 5 rows



# Создание витрины "E"
Необходимо найти телефон пользователя и захешировать его алгоритмом хешировани **md5**. Делается это с помощью встроеной функции "pyspark.sql.functions.md5"

In [22]:
data_mart_e = initial_data_df.select(
    (F.monotonically_increasing_id() + 1).alias("ID"),
    F.md5(search_values_from_maptype("raw_cookie", "user_phone"))
      .alias("hash_phone_md5")
)

data_mart_e.show(5, truncate=False)


+---+--------------------------------+
|ID |hash_phone_md5                  |
+---+--------------------------------+
|1  |5cf8badf3d5062979777ac594815ba79|
|2  |bb1200d300efac2ded3df96aafca2888|
|3  |73b52c706feaddca3655386a18a289e3|
|4  |1470e3ae80a117cbad149d3cd5d9da2d|
|5  |3d0a3adcfae1aaec5626a9bd4d558d24|
+---+--------------------------------+
only showing top 5 rows



# Создание витрины "F"
Тоже самое что и в витрине "E", только с полем "user_mail".

In [23]:
data_mart_f = initial_data_df.select(
    (F.monotonically_increasing_id() + 1).alias("ID"),
    F.md5(search_values_from_maptype("raw_cookie", "user_mail"))
      .alias("hash_email_md5")
)

data_mart_f.show(5, truncate=False)

+---+--------------------------------+
|ID |hash_email_md5                  |
+---+--------------------------------+
|1  |09eda261995f5abb1da4e12e8b944a0a|
|2  |c11db83683e26bc06bb7a499d11be20f|
|3  |02ffeb46507c4a4124b282ccb0ea8380|
|4  |5cc5536fe7783f22fe70812e1390dfef|
|5  |f41ca8ef4f8ea349c96c23121b75933c|
+---+--------------------------------+
only showing top 5 rows



# Создание витрины "G"

Замена значения через функцию "match_code".

In [24]:
data_mart_g = initial_data_df.select(
    (F.monotonically_increasing_id() + 1).alias("ID"),
    search_values_from_maptype("raw_cookie", "user_uid")
      .alias("user_uid"),
    match_code("user_os").alias("match_code")
)

data_mart_g.show(5, truncate=False)

+---+--------+----------+
|ID |user_uid|match_code|
+---+--------+----------+
|1  |7790211 |NULL      |
|2  |0747053 |NULL      |
|3  |0731034 |NULL      |
|4  |5485083 |NULL      |
|5  |2968111 |NULL      |
+---+--------+----------+
only showing top 5 rows



# Создание обобщенной витрины объединяющая все предыдущие витрины



In [25]:
main_marts = (
    data_mart_a.select('raw_cookie','data_value').alias("df_a")
    .join(
        data_mart_d.alias("df_d"),
        on=(
            (F.md5(F.col("df_d.inn")) == F.col("df_a.data_value")) |
            (F.sha2(F.col("df_d.inn"), 256) == F.col("df_a.data_value"))
            ),
        how='right'
      )
    .join(
        data_mart_b.alias("df_b"),
        on=(
            (F.md5(F.col("df_b.inn")) == F.col("df_a.data_value")) |
            (F.sha2(F.col("df_b.inn"), 256) == F.col("df_a.data_value"))
           ),
        how="left"
      )
    .join(
        data_mart_e.alias("df_e"),
        on = (
           F.md5(search_values_from_maptype("raw_cookie",
                                "user_phone")) == F.col("df_e.hash_phone_md5")),
        how ="left"
      )
    .join(
        data_mart_f.alias('df_f'),
        on = (
            F.md5(search_values_from_maptype("raw_cookie",
                                 "user_mail")) == F.col("df_f.hash_email_md5")),
        how = "left"
      )
    .join(
        data_mart_g.alias("df_g"),
        on = (
              search_values_from_maptype("raw_cookie",
                                         "user_uid") == F.col("df_g.user_uid")),
        how = "left"
      )
    .join(
        data_mart_c.alias('df_c'),
        on = (search_values_from_maptype("raw_cookie",
                                  "_sa_cookie_a") == F.col("df_c.sa_cookie_a")),
        how = "left"
      )
    .select(F.col("df_g.user_uid"),
            F.col("df_d.inn").alias("inn"),
            F.col("df_a.data_value").alias("inn_hash"),
            F.col("df_e.hash_phone_md5"),
            F.col("df_f.hash_email_md5"),
            search_values_from_maptype("raw_cookie",
                                       "user_phone").alias("user_phone"),
            search_values_from_maptype("raw_cookie",
                                       "user_mail").alias("user_email"),
            search_values_from_maptype("raw_cookie",
                                       "org_uid").alias("org_uid"),
            F.col("df_d.id").alias("id_d"),
            F.col("df_c.id").alias("id_c"),
            F.col("df_b.id").alias("id_b"),
            F.col("df_e.id").alias("id_e"),
            F.col("df_f.id").alias("id_f"),
            F.col("df_g.id").alias("id_g"),
            F.array(
                search_values_from_maptype("raw_cookie", "_sa_cookie_a"),
                search_values_from_maptype("raw_cookie", "_fa_cookie_a"),
                search_values_from_maptype("raw_cookie", "_ym_cookie_c"),
                search_values_from_maptype("raw_cookie", "_fbp")
            ).alias("array_coockie")
     )
)
main_marts.show(10, truncate=False)

+--------+------------+----------------------------------------------------------------+--------------------------------+--------------------------------+-----------+------------------+-------+----+----+----+----+----+----+---------------------------------------------------------------------------------------------------------------------------------------+
|user_uid|inn         |inn_hash                                                        |hash_phone_md5                  |hash_email_md5                  |user_phone |user_email        |org_uid|id_d|id_c|id_b|id_e|id_f|id_g|array_coockie                                                                                                                          |
+--------+------------+----------------------------------------------------------------+--------------------------------+--------------------------------+-----------+------------------+-------+----+----+----+----+----+----+---------------------------------------------------------

### Сохранение итоговой витрины

В конце пайплайна я сохраняю итоговую витрину `main_marts` в **Parquet** на Google Drive.  
Parquet выбрал потому что это удобный формат для витрин: он компактный, сохраняет типы данных и быстрее читается обратно в Spark.

Режим `overwrite` использую, чтобы при повторном запуске ноутбука результат пересчитывался и перезаписывался без ручной очистки папок.


In [26]:
OUT_MAIN = "/content/gdrive/MyDrive/data/mart/main_marts_parquet"

(main_marts
 .write
 .mode("overwrite")
 .parquet(OUT_MAIN))
